# WebGL Scatter (200k interactive points)

**Family:** Interactive, animated & big data &nbsp;|&nbsp; **Library:** plotly &nbsp;|&nbsp; **Level:** Advanced

> *Interactive, animated & big data:* What-if sliders, linked views, animation and million-point rendering. Use these when exploration or motion adds understanding that a static chart cannot.

## Design purpose

An interactive scatter drawn on the GPU (WebGL) instead of as SVG elements, so a browser can pan, zoom and hover over hundreds of thousands of points smoothly. It keeps the individual point and its hover details, which density images like datashader give up.

## When to use

- Explore 50k to ~1M points interactively, zooming into dense regions
- Points must stay individually inspectable (hover to see a trade, a customer, a sensor reading)
- Share an exploratory view as a standalone HTML file

## Avoid when

- Millions of points or more (the file gets heavy; aggregate with datashader)
- Static print output (use a density plot)
- Small data (normal SVG scatter is crisper)

## Alternatives

- Million-point scatter with datashader (static density)
- Hexbin plot

## Design tips

- Use small markers with some transparency; zoom does the rest
- Store data as float32 to keep the HTML file small
- Keep to ≤ 3 color groups on a scatter
- Plotly switches to WebGL with `go.Scattergl` (or `render_mode="webgl"` in plotly express)

## Example

200,000 simulated stock-day observations: daily return vs. trading volume relative to its 20-day average, split by market-cap group (large, mid, small).

### Setup

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]
pio.templates["vizlib"] = go.layout.Template(layout=dict(
    colorway=PALETTE, font=dict(family="Helvetica, Arial, sans-serif", size=13, color="#0b0b0b"),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", title=dict(x=0.02, xanchor="left"),
    xaxis=dict(gridcolor="#e6e5e1", linecolor="#c9c8c3", zeroline=False),
    yaxis=dict(gridcolor="#e6e5e1", linecolor="#c9c8c3", zeroline=False),
))
pio.templates.default = "plotly_white+vizlib"
pio.renderers.default = "plotly_mimetype+notebook_connected"

### Data

In [2]:
rng = np.random.default_rng(31)
groups = {"Large cap": (100_000, 0.012), "Mid cap": (60_000, 0.018), "Small cap": (40_000, 0.026)}
frames = []
for name, (n, vol) in groups.items():
    rel_volume = rng.lognormal(0, 0.45, n)                       # volume / 20-day average
    ret = rng.standard_t(4, n) * vol * np.sqrt(rel_volume) / np.sqrt(2)
    frames.append(pd.DataFrame({"group": name, "rel_volume": rel_volume.astype("float32"),
                                "ret_pct": (100 * ret).astype("float32")}))
df = pd.concat(frames, ignore_index=True)
print(f"{len(df):,} points")
df.groupby("group")["ret_pct"].describe().round(2)

200,000 points


,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
Large cap,100000.0,0.01,1.27,-30.53,-0.62,-0.0,0.63,22.51
Mid cap,60000.0,-0.01,1.90,-25.68,-0.94,-0.0,0.93,28.28
Small cap,40000.0,-0.01,2.73,-27.42,-1.34,0.0,1.35,53.29


### Plot

In [3]:
fig = go.Figure()
# Widest group first so it sits underneath; colors stay tied to the group, not the draw order.
for i, (name, part) in reversed(list(enumerate(df.groupby("group", sort=False)))):
    fig.add_trace(go.Scattergl(
        x=part["rel_volume"].to_numpy(), y=part["ret_pct"].to_numpy(), mode="markers", name=name,
        marker=dict(size=3, color=PALETTE[i], opacity=0.35),
        hovertemplate=f"{name}<br>relative volume %{{x:.2f}}×<br>return %{{y:+.2f}}%<extra></extra>"))
fig.update_layout(
    title=f"Daily return vs. relative volume: {len(df):,} stock-days (WebGL)",
    xaxis=dict(title="Volume relative to 20-day average (log scale)", type="log"),
    yaxis=dict(title="Daily return (%)", range=[-15, 15]),
    legend=dict(itemsizing="constant", traceorder="reversed"), height=580,
)
fig.show()

## How to read it

Zoom into the center to see individual days, and hover over a point for its values. Returns fan out as relative volume rises: big moves come with heavy trading, and small caps (green) spread widest.

---
Standalone script: [`webgl_scatter.py`](webgl_scatter.py) &nbsp;·&nbsp; [Catalogue](../../README.md) &nbsp;·&nbsp; [Gallery](../../gallery/index.html)